# SEC Form 13F–CRSP mapping audit

This notebook audits the CRSP security-history input and the point-in-time CUSIP-to-PERMNO mapping produced by `scripts/06_map_sec_13f_to_crsp.py`.

It checks:

1. the structure and temporal validity of the CRSP security history;
2. completeness and internal consistency of the mapping output;
3. matched coverage across securities, manager positions, and reported value;
4. the composition of unmatched identifiers; and
5. the CRSP classification used to define the common-equity universe.

The notebook reads saved data only and does not recreate the mapping.

## 1. Setup and inputs

In [1]:
import sys
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow.parquet as pq


project_root = Path.cwd().resolve()

while not (project_root / "config" / "paths.yaml").is_file():
    if project_root.parent == project_root:
        raise FileNotFoundError("Could not locate the project root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.configuration import get_crsp_config, get_paths_config


paths = get_paths_config()
crsp = get_crsp_config()
security_history_path = paths.raw / "crsp" / "security_history.parquet"
positions_path = (
    paths.interim
    / "sec_13f"
    / "position_universe"
    / "manager_security_positions.parquet"
)
mapping_path = paths.interim / "crsp" / "security_mapping.parquet"
mapping_audit_path = paths.interim / "crsp" / "security_mapping_audit.parquet"

for path in (
    security_history_path,
    positions_path,
    mapping_path,
    mapping_audit_path,
):
    if not path.is_file():
        raise FileNotFoundError(f"Required input not found: {path}")

mapping_audit = pd.read_parquet(mapping_audit_path)
connection = duckdb.connect(database=":memory:")
connection.read_parquet(str(security_history_path)).create_view("security_history")
connection.read_parquet(str(positions_path)).create_view("positions")
connection.read_parquet(str(mapping_path)).create_view("security_mapping")

inventory = pd.DataFrame(
    {
        "table": ["security_history", "positions", "security_mapping", "mapping_audit"],
        "rows": [
            pq.ParquetFile(security_history_path).metadata.num_rows,
            pq.ParquetFile(positions_path).metadata.num_rows,
            pq.ParquetFile(mapping_path).metadata.num_rows,
            pq.ParquetFile(mapping_audit_path).metadata.num_rows,
        ],
    }
)

display(inventory.style.format({"rows": "{:,}"}))
print()
print(
    "Configured CRSP security-history window: "
    f"{crsp.security_history.start_date} to {crsp.security_history.end_date}"
)

,table,rows
0,security_history,"67,394"
1,positions,"60,766,163"
2,security_mapping,"874,798"
3,mapping_audit,104



Configured CRSP security-history window: 2013-03-31 to 2026-03-31


## 2. CRSP security-history integrity

The mapping requires a PERMNO, a valid security-information interval, and a nine-character historical CUSIP. Rows without a CUSIP remain part of the downloaded CRSP snapshot but cannot participate in CUSIP matching.

In [2]:
history_summary = connection.execute(
    '''
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT permno) AS permnos,
        COUNT(DISTINCT upper(trim(cusip9))) AS cusip9s,
        MIN(secinfostartdt) AS earliest_interval_start,
        MAX(secinfoenddt) AS latest_interval_end,
        COUNT(*) FILTER (WHERE permno IS NULL) AS missing_permno,
        COUNT(*) FILTER (
            WHERE secinfostartdt IS NULL OR secinfoenddt IS NULL
        ) AS missing_interval_dates,
        COUNT(*) FILTER (
            WHERE secinfostartdt > secinfoenddt
        ) AS reversed_intervals,
        COUNT(*) FILTER (WHERE cusip9 IS NULL) AS missing_cusip9,
        COUNT(*) FILTER (
            WHERE cusip9 IS NOT NULL AND length(trim(cusip9)) != 9
        ) AS invalid_cusip9_length
    FROM security_history
    '''
).fetchdf()

history_checks = pd.DataFrame(
    {
        "check": [
            "Rows with missing PERMNO",
            "Rows with missing security-information dates",
            "Rows with reversed security-information intervals",
            "Non-missing CUSIP9 values without nine characters",
        ],
        "violations": [
            history_summary.at[0, "missing_permno"],
            history_summary.at[0, "missing_interval_dates"],
            history_summary.at[0, "reversed_intervals"],
            history_summary.at[0, "invalid_cusip9_length"],
        ],
    }
)

display(history_summary)
display(history_checks.style.format({"violations": "{:,}"}))

print()
print(
    "CRSP history rows without CUSIP9: "
    f"{history_summary.at[0, 'missing_cusip9']:,}"
)

,rows,permnos,cusip9s,earliest_interval_start,latest_interval_end,missing_permno,missing_interval_dates,reversed_intervals,missing_cusip9,invalid_cusip9_length
0,67394,17946,23399,1969-12-19,2026-07-31,0,0,0,7048,0


,check,violations
0,Rows with missing PERMNO,0
1,Rows with missing security-information dates,0
2,Rows with reversed security-information intervals,0
3,Non-missing CUSIP9 values without nine characters,0



CRSP history rows without CUSIP9: 7,048


## 3. Mapping integrity

Every distinct report-quarter–CUSIP pair in the position universe should have exactly one mapping decision. A matched decision must identify one PERMNO whose CRSP interval contains that report date.

In [3]:
mapping_integrity = connection.execute(
    '''
    WITH position_keys AS (
        SELECT DISTINCT report_period, upper(trim(CUSIP)) AS CUSIP
        FROM positions
    ),
    missing_keys AS (
        SELECT COUNT(*) AS rows
        FROM position_keys
        ANTI JOIN security_mapping USING (report_period, CUSIP)
    ),
    extra_keys AS (
        SELECT COUNT(*) AS rows
        FROM security_mapping
        ANTI JOIN position_keys USING (report_period, CUSIP)
    )
    SELECT
        COUNT(*) - COUNT(DISTINCT (report_period, CUSIP))
            AS duplicate_mapping_keys,
        (SELECT rows FROM missing_keys) AS missing_mapping_keys,
        (SELECT rows FROM extra_keys) AS extra_mapping_keys,
        COUNT(*) FILTER (
            WHERE match_status NOT IN ('MATCHED', 'UNMATCHED', 'AMBIGUOUS')
        ) AS invalid_statuses,
        COUNT(*) FILTER (
            WHERE match_status = 'MATCHED'
              AND (candidate_permno_count != 1 OR permno IS NULL)
        ) AS invalid_matched_rows,
        COUNT(*) FILTER (
            WHERE match_status = 'UNMATCHED'
              AND (candidate_permno_count != 0 OR permno IS NOT NULL)
        ) AS invalid_unmatched_rows,
        COUNT(*) FILTER (
            WHERE match_status = 'AMBIGUOUS'
              AND (candidate_permno_count <= 1 OR permno IS NOT NULL)
        ) AS invalid_ambiguous_rows,
        COUNT(*) FILTER (
            WHERE match_status = 'MATCHED'
              AND report_period NOT BETWEEN secinfostartdt AND secinfoenddt
        ) AS matched_rows_outside_crsp_interval
    FROM security_mapping
    '''
).fetchdf()

mapping_checks = mapping_integrity.T.reset_index()
mapping_checks.columns = ["check", "violations"]

display(mapping_checks.style.format({"violations": "{:,}"}))
print()
print(f"Total mapping-integrity violations: {mapping_checks['violations'].sum():,}")

,check,violations
0,duplicate_mapping_keys,0
1,missing_mapping_keys,0
2,extra_mapping_keys,0
3,invalid_statuses,0
4,invalid_matched_rows,0
5,invalid_unmatched_rows,0
6,invalid_ambiguous_rows,0
7,matched_rows_outside_crsp_interval,0



Total mapping-integrity violations: 0


## 4. Mapping coverage

Security coverage weights every distinct CUSIP-quarter equally. Position coverage gives more weight to securities held by many managers. Value coverage is shown for completeness but should be interpreted cautiously because Notebook 3 identified unresolved SEC value-unit cases.

In [4]:
status_totals = (
    mapping_audit.groupby("match_status", as_index=False)[
        ["securities", "manager_positions", "position_value_usd"]
    ]
    .sum()
    .sort_values("match_status")
)

for count_column, share_column in {
    "securities": "security_share_percent",
    "manager_positions": "position_share_percent",
    "position_value_usd": "value_share_percent",
}.items():
    status_totals[share_column] = (
        status_totals[count_column] / status_totals[count_column].sum() * 100
    )

matched_quarters = (
    mapping_audit.loc[mapping_audit["match_status"].eq("MATCHED")]
    .sort_values("report_period")
    .reset_index(drop=True)
)

coverage_summary = (
    matched_quarters[
        ["security_share_percent", "position_share_percent", "value_share_percent"]
    ]
    .agg(["min", "median", "max"])
    .T
    .reset_index(names="coverage_measure")
)

display(
    status_totals.style.format(
        {
            "securities": "{:,}",
            "manager_positions": "{:,}",
            "position_value_usd": "${:,.0f}",
            "security_share_percent": "{:.2f}",
            "position_share_percent": "{:.2f}",
            "value_share_percent": "{:.2f}",
        }
    )
)
display(coverage_summary.style.format({"min": "{:.2f}", "median": "{:.2f}", "max": "{:.2f}"}))

print()
print("Lowest quarters by matched manager-position coverage")
display(
    matched_quarters.nsmallest(10, "position_share_percent")[
        [
            "report_period",
            "security_share_percent",
            "position_share_percent",
            "value_share_percent",
        ]
    ].reset_index(drop=True)
)

,match_status,securities,manager_positions,position_value_usd,security_share_percent,position_share_percent,value_share_percent
0,MATCHED,"422,935","59,224,806","$2,683,175,826,370,207",48.35,97.46,92.99
1,UNMATCHED,"451,863","1,541,357","$202,122,100,674,908",51.65,2.54,7.01


,coverage_measure,min,median,max
0,security_share_percent,38.24,50.94,55.54
1,position_share_percent,95.96,97.59,98.43
2,value_share_percent,66.98,98.38,99.75



Lowest quarters by matched manager-position coverage


,report_period,security_share_percent,position_share_percent,value_share_percent
0,2013-06-30,41.850968,95.964657,92.690879
1,2013-12-31,38.239595,96.129463,74.134500
2,2013-09-30,38.939586,96.147489,95.925452
3,2021-03-31,49.888251,96.196334,98.499093
4,2021-12-31,51.390210,96.224521,98.048386
5,2022-03-31,51.041157,96.461184,99.068816
6,2015-06-30,38.579335,96.476880,96.178924
7,2021-06-30,51.008255,96.497645,96.784885
8,2022-06-30,55.346023,96.530535,98.228438
9,2014-06-30,39.795685,96.590227,80.756199


## 5. Unmatched identifiers

Unmatched CUSIPs are summarized by their number of manager holdings and by the free-text 13F security class. These labels are descriptive only and are not used to create approximate matches.

In [5]:
connection.execute(
    '''
    CREATE TEMP VIEW mapped_positions AS
    SELECT
        positions.* EXCLUDE (CUSIP),
        upper(trim(positions.CUSIP)) AS CUSIP,
        mapping.match_status,
        mapping.permno,
        mapping.permco,
        mapping.primaryexch,
        mapping.sharetype,
        mapping.securitytype,
        mapping.securitysubtype,
        mapping.usincflg,
        mapping.issuertype
    FROM positions
    INNER JOIN security_mapping AS mapping
        ON positions.report_period = mapping.report_period
       AND upper(trim(positions.CUSIP)) = mapping.CUSIP
    '''
)

unmatched_concentration = connection.execute(
    '''
    WITH security_periods AS (
        SELECT
            report_period,
            CUSIP,
            COUNT(*) AS manager_positions
        FROM mapped_positions
        WHERE match_status = 'UNMATCHED'
        GROUP BY report_period, CUSIP
    )
    SELECT
        COUNT(*) AS unmatched_security_periods,
        SUM(manager_positions) AS unmatched_manager_positions,
        COUNT(*) FILTER (WHERE manager_positions = 1)
            AS held_by_one_manager,
        MEDIAN(manager_positions) AS median_managers_per_security,
        MAX(manager_positions) AS maximum_managers_per_security
    FROM security_periods
    '''
).fetchdf()

unmatched_classes = connection.execute(
    '''
    SELECT
        COALESCE(upper(trim(TITLEOFCLASS)), '[MISSING]') AS title_of_class,
        COUNT(*) AS manager_positions,
        COUNT(DISTINCT (report_period, CUSIP)) AS security_periods
    FROM mapped_positions
    WHERE match_status = 'UNMATCHED'
    GROUP BY title_of_class
    ORDER BY manager_positions DESC
    LIMIT 12
    '''
).fetchdf()

display(unmatched_concentration.style.format({
    "unmatched_security_periods": "{:,}",
    "unmatched_manager_positions": "{:,}",
    "held_by_one_manager": "{:,}",
    "median_managers_per_security": "{:.1f}",
    "maximum_managers_per_security": "{:,}",
}))
display(unmatched_classes.style.format({"manager_positions": "{:,}", "security_periods": "{:,}"}))

,unmatched_security_periods,unmatched_manager_positions,held_by_one_manager,median_managers_per_security,maximum_managers_per_security
0,"451,863","1,541,357.0","301,373",1.0,491


,title_of_class,manager_positions,security_periods
0,COM,"365,269","164,727"
1,UNIT 99/99/9999,"62,380","3,710"
2,ADR,"38,330","14,281"
3,COMMON STOCK,"33,164","25,275"
4,*W EXP 99/99/999,"27,743","1,438"
5,COMMON,"22,277","15,685"
6,SPON ADR REP PFD,"19,382",146
7,PFD,"17,836","8,741"
8,PREFERRED STOCK,"12,150","5,179"
9,SPONSORED ADR,"11,971","1,730"


## 6. CRSP common-equity classification

The next stage retains exact matches classified by CRSP as U.S. common equity on the NYSE, NYSE American, or Nasdaq. CRSP classifications, rather than 13F free-text labels, define this universe.

In [6]:
common_equity_summary = connection.execute(
    '''
    SELECT
        CASE
            WHEN match_status = 'MATCHED'
             AND securitytype = 'EQTY'
             AND securitysubtype = 'COM'
             AND sharetype = 'NS'
             AND usincflg = 'Y'
             AND primaryexch IN ('N', 'A', 'Q')
                THEN 'ELIGIBLE_US_COMMON_EQUITY'
            WHEN match_status = 'MATCHED'
                THEN 'EXCLUDED_MATCHED_SECURITY'
            ELSE 'UNMATCHED'
        END AS status,
        COUNT(*) AS manager_positions,
        COUNT(DISTINCT (report_period, CUSIP)) AS security_periods,
        COUNT(DISTINCT CIK) AS managers
    FROM mapped_positions
    GROUP BY status
    ORDER BY status
    '''
).fetchdf()

connection.close()

display(
    common_equity_summary.style.format(
        {
            "manager_positions": "{:,}",
            "security_periods": "{:,}",
            "managers": "{:,}",
        }
    )
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,status,manager_positions,security_periods,managers
0,ELIGIBLE_US_COMMON_EQUITY,"40,415,180","206,868","12,211"
1,EXCLUDED_MATCHED_SECURITY,"18,809,626","216,067","11,998"
2,UNMATCHED,"1,541,357","451,863","8,034"


## 7. Interpretation

- **The CRSP input is structurally valid.** Its 67,394 history rows contain no missing PERMNOs, missing or reversed information intervals, or malformed non-missing CUSIP9 values. The 7,048 rows without CUSIP9 cannot participate in CUSIP matching and are correctly ignored. Intervals can begin before or end after the configured window because the download retains any history row that overlaps that window.
- **The point-in-time mapping is internally complete.** All 874,798 report-quarter–CUSIP keys in the position universe receive exactly one mapping decision. There are no duplicate, missing, extra, temporally invalid, or ambiguous mapping rows.
- **Coverage is high for economically recurring holdings.** Exact CRSP matches cover 48.35% of distinct security-period identifiers but 97.46% of the 60.8 million manager-position rows. Of the 451,863 unmatched security-periods, 301,373 are held by only one manager. The low identifier rate therefore reflects a long tail of sparsely held instruments rather than broad loss of institutional holdings.
- **Free-text 13F labels should not determine eligibility.** Unmatched observations include labels such as `COM`, ADRs, preferred shares, units, and warrants. Because these labels are heterogeneous, introducing approximate matches or filtering by the 13F label would create avoidable classification error.
- **The CRSP common-equity rule is selective by design.** Of the matched positions, 40,415,180 satisfy the U.S. common-equity criteria, covering 206,868 security-periods and 12,211 managers. The remaining matched securities are excluded by CRSP type, incorporation, or exchange classifications in the next construction stage.
- **Value-weighted coverage is descriptive.** Overall matched reported-value coverage is 92.99%, but quarter-specific values inherit the unresolved SEC unit cases identified in Notebook 3. Manager-position coverage is therefore the cleaner mapping-quality measure.

The mapping is suitable for constructing the final SEC Form 13F–CRSP holdings panel. Exact point-in-time matches should be retained, with no approximate CUSIP fallback.